# Clasificador de identidad y emoción (TF/Keras)

## 1. Setup

In [ ]:
import os, sys, shutil, subprocess, warnings, zipfile
from pathlib import Path

USAR_DRIVE = False   # True = leer el proyecto entero desde MyDrive/IA3_TP1 (lento: copia ~2700 fotos de a una)
DRIVE_RESPALDO = True  # monta Drive solo como 'pendrive': trae MyDrive/datos_tp1.zip y respalda results_tp1.zip ahí
DATA_ZIP_URL = ''    # modo SIN Drive: link o ID de Drive de datos_tp1.zip (vacío = subirlo a /content/datos_tp1.zip)
CONTENT = Path(os.environ.get('TP1_CONTENT', '/content'))   # disco local de Colab

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False


def extraer_zip_datos(zip_path, root):
    # Descomprime datos_tp1.zip (datos/, src/, splits/) en root de forma atómica (carpeta temporal + rename).
    # Se vuelve a extraer si el zip es más nuevo que la última extracción (p. ej. cambiaste el código).
    zip_path, root = Path(zip_path), Path(root)
    sello, mt = root / '.zip_mtime', zip_path.stat().st_mtime
    if (root / 'datos').exists() and (root / 'src' / 'tp1').exists() and sello.exists() and float(sello.read_text()) >= mt:
        return
    tmp = root.parent / (root.name + '_tmp')
    shutil.rmtree(tmp, ignore_errors=True)
    tmp.mkdir(parents=True)
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(tmp)
    root.mkdir(parents=True, exist_ok=True)
    for item in tmp.iterdir():
        shutil.rmtree(root / item.name, ignore_errors=True)
        item.rename(root / item.name)
    shutil.rmtree(tmp, ignore_errors=True)
    sello.write_text(str(mt))


if IN_COLAB:
    os.environ['TP1_COLAB'] = '1'
    drive_root = CONTENT / 'drive' / 'MyDrive'
    DRIVE_OK = drive_root.exists()
    if (USAR_DRIVE or DRIVE_RESPALDO) and not DRIVE_OK:
        try:
            from google.colab import drive
            drive.mount(str(CONTENT / 'drive'))
            DRIVE_OK = drive_root.exists()
        except Exception as e:   # p. ej. extensión de VS Code sin soporte de Drive
            print('No se pudo montar Drive (', type(e).__name__, '). Se usa el modo sin Drive (zip).')
    if not USAR_DRIVE:
        DRIVE_OK = False
    if DRIVE_OK and not os.environ.get('TP1_ROOT') and not (drive_root / 'IA3_TP1' / 'src' / 'tp1').exists():
        print('Drive montado, pero no existe MyDrive/IA3_TP1/src/tp1: subí la carpeta del proyecto con ese nombre. '
              'Mientras tanto se usa el modo sin Drive (zip).')
        DRIVE_OK = False
    os.environ['TP1_DRIVE'] = '1' if DRIVE_OK else '0'
    ROOT = Path(os.environ.get('TP1_ROOT') or (drive_root / 'IA3_TP1' if DRIVE_OK else CONTENT / 'tp1_work'))
    if not DRIVE_OK:
        ZIP = CONTENT / 'datos_tp1.zip'
        if not ZIP.exists() and (drive_root / 'datos_tp1.zip').exists():   # subido a Mi unidad
            print('Copiando datos_tp1.zip desde MyDrive...')
            shutil.copy(drive_root / 'datos_tp1.zip', ZIP)
        if not ZIP.exists() and DATA_ZIP_URL:
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'gdown'], check=True)
            import gdown
            gdown.download(DATA_ZIP_URL if DATA_ZIP_URL.startswith('http') else f'https://drive.google.com/uc?id={DATA_ZIP_URL}',
                           str(ZIP), quiet=False, fuzzy=True)
        if not ZIP.exists():
            raise FileNotFoundError(f'Sin Drive y sin {ZIP}: subí datos_tp1.zip desde el explorador de archivos '
                                    'de la extensión o completá DATA_ZIP_URL (ver README).')
        extraer_zip_datos(ZIP, ROOT)
        RZ = CONTENT / 'results_tp1.zip'    # resultados de una sesión anterior (para retomar)
        RES = ROOT / ('results_quick' if os.environ.get('TP1_QUICK') == '1' else 'results')
        if not RZ.exists() and (drive_root / 'results_tp1.zip').exists():   # respaldo de una sesión anterior
            shutil.copy(drive_root / 'results_tp1.zip', RZ)
        if RZ.exists() and not RES.exists():
            with zipfile.ZipFile(RZ) as z:
                z.extractall(RES)
            print('Resultados anteriores restaurados desde', RZ)
    RESPALDO = drive_root if (not DRIVE_OK and DRIVE_RESPALDO and drive_root.exists()) else None
else:
    DRIVE_OK, RESPALDO = False, None
    ROOT = Path(os.environ.get('TP1_ROOT', Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()))
os.environ['TP1_ROOT'] = str(ROOT)
sys.path.insert(0, str(ROOT / 'src'))

try:
    import imagehash  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'imagehash'], check=True)

import numpy as np, pandas as pd, matplotlib.pyplot as plt
import tensorflow as tf
tf.keras.utils.set_random_seed(42)
# tf.config.experimental.enable_op_determinism()  # más reproducible pero bastante más lento

gpus = tf.config.list_physical_devices('GPU')
print('TensorFlow', tf.__version__, '| GPUs:', gpus)
if gpus and shutil.which('nvidia-smi'):
    subprocess.run(['nvidia-smi'])
elif not gpus:
    print('ATENCIÓN: sin GPU. En Colab: Entorno de ejecución > Cambiar tipo > T4 GPU.')

from tp1 import config, data, pipeline, models, train, evaluate, analysis, atajos
config.ensure_local_data()   # valida los datos; con Drive los copia a /content/data (lectura rápida)
print('Raíz del proyecto:', config.PROJECT_ROOT, '| Drive:', DRIVE_OK, '| QUICK =', config.QUICK)
print('Resultados en:', config.RESULTS_DIR)
if config.NO_DRIVE:
    if RESPALDO is not None:
        print('Modo zip con respaldo: results_tp1.zip se copia a', RESPALDO, 'después de cada modelo.')
    else:
        print('AVISO: sin Drive. Si se corta la sesión se pierden los checkpoints; descargá', config.RESULTS_ZIP)


def respaldar_resultados():
    # Sin Drive: zip de results/ en /content y, si Drive está montado, una copia en MyDrive (sobrevive a cortes)
    z = config.zip_results()
    if RESPALDO is not None:
        shutil.copy(z, RESPALDO / z.name)
        print('Resultados respaldados en', RESPALDO / z.name)
    else:
        print('Zip de resultados actualizado (descargalo):', z)

In [ ]:
DATASET = os.environ.get('TP1_DATASET', 'g8')      # 'final' | 'g8'
TAREAS = ['identidad', 'emocion']
MODELOS = ['cnn', 'cnn_residual', 'cnn_separable', 'cnn_inception',
           'mobilenetv2', 'efficientnetb0', 'resnet50v2']   # agregar 'mlp' al principio si se quiere

BATCH = 32 if DATASET == 'final' else 16
WEIGHTS = None if config.QUICK else 'imagenet'         # QUICK local: sin descargar pesos
EPOCAS_PROPIOS, EPOCAS_FASE1, EPOCAS_FASE2, PACIENCIA = (2, 1, 1, 2) if config.QUICK else (60, 15, 25, 6)
PACIENCIA_PROPIOS = 2 if config.QUICK else 15   # las CNNs desde cero arrancan más inestables: más paciencia
FORZAR_REENTRENAR = False                               # True ignora lo ya entrenado
REGENERAR_PARTICION = False                             # True recalcula la partición aunque exista splits/splits_*.csv

assert DATASET in ('final', 'g8')
TASK_COL = {'identidad': 'actor', 'emocion': 'emocion'}
print(train.setup_precision(), '| dataset:', DATASET, '| batch:', BATCH)

## 2. Datos, duplicados y partición reproducible

In [ ]:
# Usa la partición guardada (splits/) si coincide con los archivos; si no, la calcula y la guarda
df, SPLIT_HASH, reporte = data.ensure_splits(DATASET, regenerar=REGENERAR_PARTICION)
print('Hash de la partición:', SPLIT_HASH[:8], '(archivo:', config.splits_path(DATASET).name + ')')
if len(reporte):
    print('Duplicados exactos (md5) encontrados:')
    display(reporte)
print(f'\nDataset "{DATASET}": {len(df)} imágenes, {df.actor.nunique()} actores')

In [ ]:
# Verificación breve: conteos por clase y por partición
for col in ('actor', 'emocion'):
    t = pd.crosstab(df[col], df['split'])[['train', 'val', 'test']]
    t['total'] = t.sum(axis=1)
    display(t.sort_values('total', ascending=False))

prop = df['split'].value_counts(normalize=True).round(3)
print('Proporciones:', prop.to_dict())
n_multi = (df.groupby('grupo_dup')['split'].nunique() > 1).sum()
print('Grupos de casi-duplicados repartidos en más de una partición:', n_multi, '(debe ser 0)')
print('Tamaño del grupo más grande:', df['grupo_dup'].value_counts().max())

In [ ]:
# V de Cramér actor <-> emoción sobre TODO el dataset + proporción de emociones por actor
config.RESULTS_DIR.mkdir(parents=True, exist_ok=True)
cv = atajos.cramers_v(df['actor'], df['emocion'])
print(f"V de Cramér actor <-> emoción = {cv['v']:.3f}   (chi2 = {cv['chi2']:.1f}, gl = {cv['gl']}, n = {cv['n']})")
atajos.plot_emocion_por_actor(df, path=config.RESULTS_DIR / f'emocion_por_actor_{DATASET}.png')
plt.show()

## 3. Pipeline `tf.data` y augmentación

In [ ]:
# Originales vs. aumentadas (las mismas 4 imágenes pasadas 3 veces por el aumentador)
aug = pipeline.get_augmenter()
muestra = df[df.split == 'train'].sample(4, random_state=0)
orig = np.stack([np.asarray(plt.imread(p)) for p in muestra.path]).astype('float32')
if orig.max() <= 1.0:
    orig *= 255
fig, axs = plt.subplots(4, 4, figsize=(9, 9))
for j in range(4):
    axs[0, j].imshow(orig[j].astype('uint8')); axs[0, j].set_title('original', fontsize=8)
    for i in range(1, 4):
        axs[i, j].imshow(np.clip(aug(orig[j:j+1], training=True)[0].numpy(), 0, 255).astype('uint8'))
        axs[i, j].set_title(f'aumentada {i}', fontsize=8)
for a in axs.ravel(): a.axis('off')
plt.tight_layout(); plt.show()

## 4. Baseline simple

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier

X = {s: data.load_gray_pixels(df[df.split == s].path) for s in ('train', 'val', 'test')}
A = {s: df[df.split == s]['actor'].to_numpy() for s in ('train', 'val', 'test')}   # el baseline 'solo actor' usa SOLO esto


def predecir(m, s):
    # 'solo actor' recibe el actor de cada fila; el resto de los baselines, los píxeles
    return m.predict(A[s] if isinstance(m, atajos.SoloActor) else X[s])


base_models, base_val = {}, []          # (tarea, nombre) -> modelo ya entrenado
for tarea in TAREAS:
    col = TASK_COL[tarea]
    clases, idx = pipeline.label_maps(df, col)
    y = {s: df[df.split == s][col].map(idx).to_numpy() for s in ('train', 'val', 'test')}
    cand = {'dummy': DummyClassifier(strategy='most_frequent').fit(X['train'], y['train'])}
    mejor = None
    for depth in (None, 20):
        rf = RandomForestClassifier(n_estimators=100 if config.QUICK else 300, max_depth=depth,
                                    class_weight='balanced', n_jobs=-1, random_state=config.SEED)
        rf.fit(X['train'], y['train'])
        f1 = evaluate.compute_metrics(y['val'], rf.predict(X['val']))['f1_macro']
        if mejor is None or f1 > mejor[0]:
            mejor = (f1, depth, rf)
    cand['random_forest'] = mejor[2]
    if tarea == 'emocion':
        cand['solo_actor'] = atajos.SoloActor().fit(A['train'], y['train'])   # control de atajo: sin mirar la imagen
    print(f'{tarea}: mejor max_depth del RF = {mejor[1]}')
    for nombre, m in cand.items():
        base_models[(tarea, nombre)] = m
        base_val.append({'tarea': tarea, 'modelo': nombre,
                         **evaluate.compute_metrics(y['val'], predecir(m, 'val'))})
display(pd.DataFrame(base_val).round(3))

## 5. Modelos

In [ ]:
# Las 4 CNNs propias sin entrenar (solo se construyen): tamaño, profundidad y mapa final, con su "por qué"
n_id = df['actor'].nunique()
tabla_cnns = analysis.compare_cnns({m: models.build_model(m, config.MODEL_IMG_SIZE[m], n_id, None) for m in config.CNNS_PROPIAS})
with pd.option_context('display.max_colwidth', None):
    display(tabla_cnns)

In [ ]:
image_cache, ds_cache, metas, clases_de, dirs = {}, {}, {}, {}, {}
for tarea in TAREAS:
    col = TASK_COL[tarea]
    clases, idx = pipeline.label_maps(df, col)
    clases_de[tarea] = clases
    y_train = df[df.split == 'train'][col].map(idx).to_numpy()
    cw = train.balanced_class_weights(y_train, len(clases))
    print(f'\n===== {tarea}: {len(clases)} clases =====')
    for modelo in MODELOS:
        size = config.MODEL_IMG_SIZE[modelo]
        if (tarea, size) not in ds_cache:   # las imágenes decodificadas se comparten entre tareas (image_cache)
            ds_cache[(tarea, size)] = pipeline.make_datasets(df, col, size, BATCH, image_cache)
        ds_tr, ds_va, _, _ = ds_cache[(tarea, size)]
        out = dirs[(tarea, modelo)] = config.model_dir(DATASET, SPLIT_HASH, tarea, modelo)
        meta = train.train_model(
            out, modelo, lambda m=modelo, s=size, n=len(clases): models.build_model(m, s, n, WEIGHTS),
            ds_tr, ds_va, cw,
            extra={'tarea': tarea, 'dataset': DATASET, 'img_size': size, 'batch': BATCH},
            epochs=EPOCAS_PROPIOS if modelo in config.MODELOS_PROPIOS else EPOCAS_FASE1,
            epochs_ft=EPOCAS_FASE2, patience=PACIENCIA_PROPIOS if modelo in config.MODELOS_PROPIOS else PACIENCIA, force=FORZAR_REENTRENAR, split_hash=SPLIT_HASH)
        metas[(tarea, modelo)] = meta
        evaluate.plot_history(out / 'history.csv', f'{tarea} / {modelo}', out / 'curvas.png',
                              fase_boundary=meta.get('fase1_epocas'))
        plt.show()
        if config.NO_DRIVE:   # sin Drive: zip de resultados después de cada modelo (+ copia en MyDrive)
            respaldar_resultados()

## 6. Evaluación en validación

In [ ]:
loaded, val_rows, val_probs = {}, [], {}
for tarea in TAREAS:
    col = TASK_COL[tarea]
    for modelo in MODELOS:
        size = config.MODEL_IMG_SIZE[modelo]
        m = tf.keras.models.load_model(dirs[(tarea, modelo)] / 'best.keras')
        loaded[(tarea, modelo)] = m
        probs, y = evaluate.predict_probs(m, ds_cache[(tarea, size)][1])
        val_probs[(tarea, modelo)] = probs
        meta = metas[(tarea, modelo)]
        val_rows.append({'tarea': tarea, 'modelo': modelo, **evaluate.compute_metrics(y, probs.argmax(1)),
                         'params': meta['params_total'], 'tiempo_s': meta['train_seconds'],
                         'epocas': meta['epocas_corridas']})
val_df = pd.DataFrame(val_rows)
elegido = {t: val_df[val_df.tarea == t].sort_values('f1_macro', ascending=False).iloc[0]['modelo'] for t in TAREAS}
display(val_df.round(3))
print('Modelo elegido por F1 macro en validación:', elegido)

## 7. Test: se evalúa una sola vez

In [ ]:
test_rows, test_probs, y_test = [], {}, {}
for tarea in TAREAS:
    col = TASK_COL[tarea]
    clases, idx = pipeline.label_maps(df, col)
    yt = df[df.split == 'test'][col].map(idx).to_numpy()
    y_test[tarea] = yt
    for (t, nombre), m in base_models.items():
        if t != tarea: continue
        p = predecir(m, 'test')
        test_rows.append({'tarea': tarea, 'modelo': nombre, 'familia': 'baseline',
                          **evaluate.compute_metrics(yt, p), 'params': np.nan, 'tiempo_s': np.nan})
    for modelo in MODELOS:
        size = config.MODEL_IMG_SIZE[modelo]
        probs, y = evaluate.predict_probs(loaded[(tarea, modelo)], ds_cache[(tarea, size)][2])
        assert (y == yt).all()
        test_probs[(tarea, modelo)] = probs
        meta = metas[(tarea, modelo)]
        fam = 'desde cero' if modelo in config.MODELOS_PROPIOS else 'transferencia'
        test_rows.append({'tarea': tarea, 'modelo': modelo, 'familia': fam,
                          **evaluate.compute_metrics(y, probs.argmax(1)),
                          'params': meta['params_total'], 'tiempo_s': meta['train_seconds']})

tabla = pd.DataFrame(test_rows)
tabla.insert(0, 'dataset', DATASET)
tabla['val_f1_macro'] = [val_df.loc[(val_df.tarea == r.tarea) & (val_df.modelo == r.modelo), 'f1_macro'].squeeze()
                         if r.familia != 'baseline' else np.nan for r in tabla.itertuples()]
tabla['elegido'] = [elegido[r.tarea] == r.modelo for r in tabla.itertuples()]
tabla = tabla.sort_values(['tarea', 'f1_macro'], ascending=[True, False])
config.RESULTS_DIR.mkdir(parents=True, exist_ok=True)
tabla.to_csv(config.RESULTS_DIR / f'tabla_comparativa_{DATASET}.csv', index=False)
(config.RESULTS_DIR / f'tabla_comparativa_{DATASET}.md').write_text(evaluate.to_markdown(tabla), encoding='utf-8')
display(tabla.round(3))

## 8. Matrices de confusión y análisis de errores

In [ ]:
for tarea in TAREAS:
    modelo = elegido[tarea]
    clases = clases_de[tarea]
    probs, yt = test_probs[(tarea, modelo)], y_test[tarea]
    pred = probs.argmax(1)
    print(f'\n##### {tarea} - {modelo} (test) #####')
    evaluate.plot_confusion(yt, pred, clases, f'{tarea}: {modelo}',
                            dirs[(tarea, modelo)] / 'confusion_test.png')
    plt.show()
    print('Pares más confundidos (real → predicho):')
    display(evaluate.top_confusions(yt, pred, clases, k=5))
    paths = df[df.split == 'test']['path'].to_numpy()
    errs = evaluate.confident_errors(probs, yt, paths, clases, k=8)
    evaluate.plot_error_grid(errs, cols=4, path=dirs[(tarea, modelo)] / 'errores_confiados.png')
    plt.show()

In [ ]:
if 'emocion' in TAREAS and ('emocion', 'solo_actor') in base_models:
    modelo = elegido['emocion']
    sa = base_models[('emocion', 'solo_actor')]
    pred_e = test_probs[('emocion', modelo)].argmax(1)
    por_actor, resumen = atajos.diagnostico_emocion(A['test'], y_test['emocion'], pred_e, sa.tipica_, clases_de['emocion'])
    print(f'Modelo elegido para emoción: {modelo}')
    display(por_actor.round(3))
    por_actor.to_csv(config.RESULTS_DIR / f'diagnostico_emocion_por_actor_{DATASET}.csv', index=False)
    n_at, n_te = resumen['n_atipicas'], resumen['n_test']
    print(f"\nImágenes de test con emoción NO típica de su actor: {n_at} de {n_te} ({n_at / max(n_te, 1):.0%})")
    if n_at:
        print(f"  - el modelo predice igual la emoción típica del actor: {resumen['pct_predice_tipica']:.0%}")
        print(f"  - el modelo acierta la emoción real:                   {resumen['pct_acierta']:.0%}")
        print(f"  - se equivoca hacia otra emoción:                      {resumen['pct_otra']:.0%}")
    else:
        print('  (no hay imágenes atípicas en test: no se puede medir)')
    fig, ax = plt.subplots(figsize=(8, 0.3 * len(por_actor) + 1.5))
    y_pos = np.arange(len(por_actor))
    ax.barh(y_pos - 0.2, por_actor['accuracy'], 0.4, label='accuracy')
    ax.barh(y_pos + 0.2, por_actor['f1_macro'], 0.4, label='F1 macro (emociones del actor)')
    ax.set_yticks(y_pos, [f"{a} (n={n})" for a, n in zip(por_actor['actor'], por_actor['n'])])
    ax.set_xlim(0, 1); ax.invert_yaxis(); ax.legend(loc='lower right'); ax.set_title(f'Emoción por actor en test - {modelo}')
    plt.tight_layout(); plt.show()
else:
    print("Esta subsección necesita la tarea 'emocion' (y el baseline 'solo_actor' de la sección 4).")

In [ ]:
# Desempeño solo sobre las filas de G8 del test (cuando se corre con el dataset final)
# f1_macro_todas_las_clases = como antes (incluye clases que el modelo predijo y no están en G8); f1_macro = solo las presentes
# labels = clases presentes en el subconjunto: el F1 macro no se promedia con actores que no están en G8
if DATASET == 'final':
    mask = df[df.split == 'test']['es_g8'].to_numpy()
    filas = []
    for tarea in TAREAS:
        modelo = elegido[tarea]
        pred = test_probs[(tarea, modelo)].argmax(1)
        if mask.sum():
            y_g8 = y_test[tarea][mask]
            filas.append({'tarea': tarea, 'modelo': modelo, 'n_g8_test': int(mask.sum()),
                          'clases_en_g8': len(np.unique(y_g8)),
                          'f1_macro_todas_las_clases': evaluate.compute_metrics(y_g8, pred[mask])['f1_macro'],
                          **evaluate.compute_metrics(y_g8, pred[mask], labels=np.unique(y_g8))})
    display(pd.DataFrame(filas).round(3))
else:
    print('Corrida sobre G8: la tabla de la sección 7 ya es solo de G8.')

## 9. ¿Sobreajuste o subajuste? Diagnóstico de las curvas

In [ ]:
diag = analysis.diagnose_all(metas, dirs, EPOCAS_PROPIOS, EPOCAS_FASE2)
if diag.empty:
    print('No hay historiales de entrenamiento utilizables: se saltea el diagnóstico.')
else:
    display(diag.round(3))
    diag.to_csv(config.RESULTS_DIR / f'diagnostico_curvas_{DATASET}.csv', index=False)
    analysis.plot_brechas(diag, config.RESULTS_DIR / f'diagnostico_brechas_{DATASET}.png')
    plt.show()

## 10. Anatomía de la CNN desde cero

In [ ]:
if ('identidad', 'cnn') in loaded:
    cnn_id = loaded[('identidad', 'cnn')]
    display(analysis.cnn_block_summary(cnn_id))
    analysis.plot_first_filters(cnn_id, path=dirs[('identidad', 'cnn')] / 'filtros_conv1.png')
    plt.show()
    X_cnn = analysis.images_from_ds(ds_cache[('identidad', config.MODEL_IMG_SIZE['cnn'])][2])
    analysis.plot_activation_maps(cnn_id, X_cnn[0], n_canales=6, path=dirs[('identidad', 'cnn')] / 'mapas_activacion.png')
    plt.show()
else:
    print("Esta sección usa la CNN desde cero: agregá 'cnn' a MODELOS.")

In [ ]:
# Tabla comparativa de las 4 CNNs propias ENTRENADAS (identidad): parámetros, profundidad, mapa final y "por qué"
propias = {m: loaded[('identidad', m)] for m in config.CNNS_PROPIAS if ('identidad', m) in loaded}
if propias:
    with pd.option_context('display.max_colwidth', None):
        display(analysis.compare_cnns(propias))
else:
    print("Agregá las CNNs propias a MODELOS para ver la comparación.")

## 11. Grad-CAM: ¿dónde mira la red?

In [ ]:
foco_rows = []
for tarea in TAREAS:
    modelo = elegido[tarea]
    if modelo == 'mlp':
        print(f'{tarea}: el modelo elegido es el MLP (sin convoluciones), Grad-CAM no aplica: se saltea.')
        continue
    m = loaded[(tarea, modelo)]
    X_img = analysis.images_from_ds(ds_cache[(tarea, config.MODEL_IMG_SIZE[modelo])][2])   # (no pisar X, el dict de la sección 4)
    # recorrido manual == model.predict: la clase tiene que coincidir; si solo difiere el redondeo fp16/XLA, avisa y sigue
    dif = analysis.verify_rebuild(m, X_img[:16], estricto=False)
    heat, probs, cls = analysis.gradcam(m, X_img)               # mapa de la clase PREDICHA
    # Con mixed precision en GPU, predict (XLA) y el recorrido manual difieren un poco:
    # lo que importa es que predigan la misma clase.
    coincide = (probs.argmax(1) == test_probs[(tarea, modelo)].argmax(1)).mean()
    assert coincide > 0.99, f'Grad-CAM no reproduce las predicciones ({coincide:.1%})'
    ok = cls == y_test[tarea]
    foco = analysis.focus_fraction(heat, region=0.6)
    foco_rows.append({'tarea': tarea, 'modelo': modelo, 'dif_max_vs_predict': dif,
                      'foco_medio': np.nanmean(foco),
                      'foco_aciertos': np.nanmean(foco[ok]) if ok.any() else np.nan,
                      'foco_errores': np.nanmean(foco[~ok]) if (~ok).any() else np.nan,
                      'referencia_uniforme': 0.6 ** 2, 'mapas_vacios': int(np.isnan(foco).sum())})
    idx = analysis.pick_examples(y_test[tarea], cls, probs, n_ok=4, n_err=4)
    clases = clases_de[tarea]
    titulos = [('ACIERTO' if ok[i] else 'ERROR') + f"\nreal: {clases[y_test[tarea][i]]} | pred: {clases[cls[i]]} ({probs[i, cls[i]]:.2f})\nfoco: {foco[i]:.2f}"
               for i in idx]
    print(f'\n##### {tarea} - {modelo} (test) #####')
    analysis.plot_gradcam_grid(X_img[idx], heat[idx], titulos, region=0.6, cols=4, path=dirs[(tarea, modelo)] / 'gradcam.png')
    plt.show()
if foco_rows:
    display(pd.DataFrame(foco_rows).round(3))

## 12. Autoencoder para revisar la calidad del dataset

In [ ]:
AE_EPOCAS, AE_PACIENCIA = (1, 2) if config.QUICK else (20, 4)
ae_dir = config.model_dir(DATASET, SPLIT_HASH, 'calidad', 'autoencoder')
ae, ae_meta = analysis.fit_autoencoder(df, ae_dir, SPLIT_HASH, size=analysis.AE_SIZE, epochs=AE_EPOCAS,
                                       patience=AE_PACIENCIA, force=FORZAR_REENTRENAR)
print(f"Autoencoder: {ae_meta['n_train']} imágenes de train, {ae_meta['n_val']} de val, {ae_meta['epocas']} épocas, val_loss = {ae_meta['val_loss']:.4f}")

x_all = analysis.load_images(df['path'], analysis.AE_SIZE)          # TODO el dataset (uint8, 64x64)
err, z_lat, rec = analysis.reconstruction_errors(ae, x_all)
calidad = analysis.quality_table(df, err)
print('Tabla de calidad guardada en', analysis.save_quality_csv(calidad, DATASET))
display(calidad.groupby('split')['error_mse'].describe().round(4))

mejores = np.argsort(err)[:3]; peores = np.argsort(-err)[:5]       # 3 mejores y 5 peores reconstruidas
analysis.plot_reconstructions(x_all, rec, list(mejores) + list(peores), path=ae_dir / 'reconstrucciones.png')
plt.show()

In [ ]:
# (a) las 16 peor reconstruidas = candidatas a recortes malos (revisar a mano)
analysis.plot_worst(calidad, k=16, cols=4, path=ae_dir / 'peores_reconstruidas.png')
plt.show()
# (b) error por split
analysis.plot_error_hist(calidad, path=ae_dir / 'error_por_split.png')
plt.show()

In [ ]:
# (c) espacio latente en 2D (PCA): por actor y por emoción
analysis.plot_latent_pca(z_lat, df['actor'], df['emocion'], path=ae_dir / 'latente_pca.png')
plt.show()
rng = np.random.default_rng(config.SEED)
for col in ('actor', 'emocion'):
    real = analysis.latent_separation(z_lat, df[col])
    azar = analysis.latent_separation(z_lat, rng.permutation(df[col].to_numpy()))   # referencia: etiquetas al azar (~0)
    print(f'Silhouette del latente (64 dim) por {col:8s}: {real:.3f}   | con etiquetas permutadas al azar: {azar:.3f}')

## 13. Cierre

In [ ]:
if config.NO_DRIVE:   # sin Drive: el disco de Colab es temporal, bajá este zip antes de cerrar la sesión
    respaldar_resultados()
else:
    print('Resultados en:', config.RESULTS_DIR)